# ChaoSSM — Phase 1b: Parity Curriculum

## Why this phase

Phase 1 tested chaos vs. a linear-SSM baseline on four tasks. Three were informative:
**copy** (chaos wins, p=0.0625, 5/5 seeds), **assoc_recall** (linear wins, p=0.0625, 5/5 seeds
— a real, mechanistically sensible negative result: chaos likely corrupts the precise
key-value addressing recall needs), **selective_copy** (weak, high-variance trend toward
chaos). **parity** — the one task designed to test genuine long-range dependency, unsolvable
by local pattern matching, and the task closest to the proposal's central novelty claim — gave
no signal: all three arms peaked well above chance (0.59–0.66) then decayed back to ~0.48–0.51,
identically regardless of architecture. That is a sparse-credit-assignment failure, not an
architecture comparison: with the only supervision at token 48 after 46 pure-noise tokens,
nothing learned the task, so parity currently says nothing about the hypothesis it exists to
test.

## Fix

A curriculum on the gap between the two signal bits: start close together (easy, short
effective range) and widen over training (hard, full 48-token range), rather than presenting
the maximum difficulty from step 0.

**Two conditions per arm**, both at the same longer step budget, to attribute any improvement
to the curriculum specifically rather than to extra training time alone:
- `curriculum=True` — gap schedule as above
- `curriculum=False` — full difficulty throughout (Phase 1's original setting, just run longer)

Same three arms (`linear`, `chaossm_free`, `chaossm_corridor`), same cell, same estimators,
same statistical protocol as Phase 1. Only the parity task and its training budget change.

## Setup

In [1]:
import os, json, math, time, warnings
from dataclasses import dataclass, asdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import wilcoxon
from torch.func import jacrev

warnings.filterwarnings("ignore")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", DEVICE)

def setup_results_dir(project="ChaoSSM", phase="phase1b"):
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        base = f"/content/drive/My Drive/{project}/results/{phase}"
    except Exception as e:
        base = os.path.abspath(f"./{project}_results/{phase}")
        print(f"[info] Not in Colab ({type(e).__name__}); using local dir.")
    os.makedirs(base, exist_ok=True); print("results ->", base); return base

RESULTS = setup_results_dir()
def savefig(f,n): p=os.path.join(RESULTS,n); f.savefig(p,dpi=300,bbox_inches="tight"); print("saved",p)
def savetable(d,n): p=os.path.join(RESULTS,n); d.to_csv(p,index=False); print("saved",p)
def savejson(o,n):
    p=os.path.join(RESULTS,n)
    with open(p,"w") as fh: json.dump(o,fh,indent=2,default=str)
    print("saved",p)

torch 2.11.0+cu128 | device: cuda
Mounted at /content/drive
results -> /content/drive/My Drive/ChaoSSM/results/phase1b


## Configuration

`steps` raised from Phase 1's 1200 to 2500 (curriculum needs runway to reach full difficulty
and then consolidate). `curriculum_frac=0.6` means the gap reaches its maximum by 60% of
training and holds there for the remaining 40%, so the model gets time to generalize to the
full-difficulty setting before evaluation.

3 arms × 2 curriculum settings × 5 seeds = 30 runs.

In [2]:
SMOKE_TEST = False   # full run: 3 arms x {curriculum, no-curriculum} x 5 seeds

@dataclass
class Config:
    state_dim: int = 48
    d_model: int = 48
    vocab: int = 16
    steps: int = 2500
    batch: int = 64
    lr: float = 3e-4
    bptt_k: int = 64
    grad_clip: float = 1.0
    gamma_init: float = 1.0
    warmup_frac: float = 0.10
    dual_lr: float = 0.05
    alpha_max: float = 50.0
    corridor_target: float = 0.15
    corridor_half_width: float = 0.10
    lyap_window: int = 4
    lyap_subsample: int = 3
    lyap_every: int = 20
    benettin_transient: int = 100
    benettin_measure: int = 250
    parity_len: int = 48
    min_gap: int = 4
    curriculum_frac: float = 0.6
    arms: tuple = ("linear", "chaossm_free", "chaossm_corridor")
    curricula: tuple = (True, False)
    seeds: int = 5
    eval_every: int = 20

CFG = Config()
if SMOKE_TEST:
    CFG.steps = 200; CFG.seeds = 1
    CFG.benettin_measure = 60; CFG.benettin_transient = 20
    print("*** SMOKE TEST — pipeline check only ***")
print(json.dumps({k: str(v) for k, v in asdict(CFG).items()}, indent=2))

{
  "state_dim": "48",
  "d_model": "48",
  "vocab": "16",
  "steps": "2500",
  "batch": "64",
  "lr": "0.0003",
  "bptt_k": "64",
  "grad_clip": "1.0",
  "gamma_init": "1.0",
  "warmup_frac": "0.1",
  "dual_lr": "0.05",
  "alpha_max": "50.0",
  "corridor_target": "0.15",
  "corridor_half_width": "0.1",
  "lyap_window": "4",
  "lyap_subsample": "3",
  "lyap_every": "20",
  "benettin_transient": "100",
  "benettin_measure": "250",
  "parity_len": "48",
  "min_gap": "4",
  "curriculum_frac": "0.6",
  "arms": "('linear', 'chaossm_free', 'chaossm_corridor')",
  "curricula": "(True, False)",
  "seeds": "5",
  "eval_every": "20"
}


## 1. Model — unchanged from Phase 1

In [3]:
def triangle_fold(x):
    return 1.0 - torch.abs(torch.remainder(x, 2.0) - 1.0)


class LinearMap(nn.Module):
    def __init__(self, N):
        super().__init__()
        W = torch.empty(N, N); nn.init.orthogonal_(W)
        self.W_lin = nn.Parameter(W * 0.5)
    def forward(self, h):
        return torch.tanh(h @ self.W_lin.T)


class ChaoticMap(nn.Module):
    '''phi(h) = fold( gamma * (W_mix h + r h(1-h)) ). sigma(J_phi) = gamma * sigma(affine)
    exactly (fold is orthogonal on derivatives) -- settled in 0d/Phase 1.'''
    def __init__(self, N, gamma_init=1.0):
        super().__init__()
        self.w_r = nn.Parameter(torch.zeros(N))
        W = torch.empty(N, N); nn.init.orthogonal_(W)
        self.W_mix = nn.Parameter(W * 0.1)
        inv = math.log(math.expm1(gamma_init))
        self.w_gamma = nn.Parameter(torch.tensor(float(inv)))
    def get_r(self): return 4.0 * torch.sigmoid(self.w_r)
    def get_gamma(self): return F.softplus(self.w_gamma)
    def forward(self, h):
        r = self.get_r()
        u = h @ self.W_mix.T + r * h * (1 - h)
        return triangle_fold(self.get_gamma() * u)


class ChaoSSMCell(nn.Module):
    def __init__(self, cfg, phi_kind):
        super().__init__()
        self.N = cfg.state_dim
        self.phi_kind = phi_kind
        self.phi = LinearMap(cfg.state_dim) if phi_kind == "linear" \
                   else ChaoticMap(cfg.state_dim, cfg.gamma_init)
        self.A = nn.Parameter(torch.diag(torch.empty(cfg.state_dim).uniform_(0.9, 1.1)))
        self.B = nn.Parameter(torch.randn(cfg.state_dim, cfg.d_model) / math.sqrt(cfg.d_model))
        self.W_g = nn.Parameter(torch.randn(cfg.state_dim, cfg.d_model) / math.sqrt(cfg.d_model))
        self.b_g = nn.Parameter(torch.zeros(cfg.state_dim))
    def step(self, h, x):
        g = torch.sigmoid(x @ self.W_g.T + self.b_g)
        raw = g * self.phi(h @ self.A.T) + (1 - g) * h + x @ self.B.T
        return triangle_fold(raw) if self.phi_kind != "linear" else torch.tanh(raw)
    def init_state(self, b, dev):
        return torch.zeros(b, self.N, device=dev) if self.phi_kind == "linear" \
               else torch.full((b, self.N), 0.5, device=dev)


class ChaoSSMLM(nn.Module):
    def __init__(self, cfg, phi_kind, vocab_size, readout_size):
        super().__init__()
        self.cfg = cfg
        self.emb = nn.Embedding(vocab_size, cfg.d_model)
        self.cell = ChaoSSMCell(cfg, phi_kind)
        self.readout = nn.Linear(cfg.state_dim, readout_size)
    def forward(self, tok, return_traj=False):
        B, T = tok.shape
        x = self.emb(tok); h = self.cell.init_state(B, tok.device)
        outs, traj = [], []
        for t in range(T):
            h = self.cell.step(h, x[:, t]); outs.append(self.readout(h))
            if return_traj: traj.append(h)
            if (t + 1) % self.cfg.bptt_k == 0: h = h.detach()
        lg = torch.stack(outs, 1)
        return (lg, torch.stack(traj, 1), x) if return_traj else lg

## 2. Estimators — unchanged from Phase 1

In [4]:
def _full_jac_fn(model, x):
    return lambda z: model.cell.step(z.unsqueeze(0), x.unsqueeze(0))[0]

def lambda_window(model, traj, xseq, W, n_sample):
    B, T, N = traj.shape
    if T <= W + 1:
        return torch.zeros((), device=traj.device)
    lams = []
    for b in torch.randperm(B)[:n_sample]:
        t0 = int(torch.randint(0, T - W - 1, (1,)))
        M = torch.eye(N, device=traj.device, dtype=traj.dtype)
        for k in range(W):
            M = jacrev(_full_jac_fn(model, xseq[b, t0+k]))(traj[b, t0+k]) @ M
        lams.append(torch.log(torch.linalg.matrix_norm(M, ord=2) + 1e-12) / W)
    return torch.stack(lams).mean()

## 3. Parity task with curriculum

`progress` is `step / (curriculum_frac * total_steps)`, clipped to [0,1]. The gap between the
two signal bits is sampled uniformly from `[min_gap, min_gap + progress*(L-2-min_gap)]` — so
early in training the two bits are close together (easy, short effective range), and by
`curriculum_frac` of the way through, the gap spans the full sequence, matching Phase 1's
original difficulty. `curriculum=False` fixes `progress=1.0` throughout, reproducing Phase 1's
task exactly, just at the longer step budget.

Position 47 (the last index) is reserved for the readout and never used as a signal-bit
position, so the model always has one full contraction step after seeing bit_b in the worst case.

In [5]:
def make_parity_batch(cfg, batch, device, gen, progress):
    L, V = cfg.parity_len, cfg.vocab
    seq = torch.randint(2, V, (batch, L), device=device, generator=gen)
    bit_a = torch.randint(0, 2, (batch,), device=device, generator=gen)
    bit_b = torch.randint(0, 2, (batch,), device=device, generator=gen)

    prog = float(np.clip(progress, 0.0, 1.0))
    max_gap = int(cfg.min_gap + prog * (L - 2 - cfg.min_gap))
    max_gap = max(cfg.min_gap, min(max_gap, L - 2))

    pos_a = torch.empty(batch, dtype=torch.long, device=device)
    pos_b = torch.empty(batch, dtype=torch.long, device=device)
    for b in range(batch):
        gap = int(torch.randint(cfg.min_gap, max_gap + 1, (1,), device=device, generator=gen))
        start_max = (L - 2) - gap                       # keep pos_b <= L-2, reserve L-1 for readout
        start_max = max(0, start_max)
        a = int(torch.randint(0, start_max + 1, (1,), device=device, generator=gen))
        pos_a[b] = a; pos_b[b] = a + gap

    bidx = torch.arange(batch, device=device)
    seq[bidx, pos_a] = bit_a
    seq[bidx, pos_b] = bit_b
    tgt = torch.full((batch, L), -100, device=device)
    tgt[:, -1] = (bit_a ^ bit_b)
    return seq, tgt


VOCAB_SIZE, READOUT_SIZE = CFG.vocab, 2

## 4. Trainer

In [6]:
def train_one(arm, use_curriculum, seed, cfg):
    torch.manual_seed(seed); np.random.seed(seed)
    gen = torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); gen.manual_seed(seed)
    phi_kind = "linear" if arm == "linear" else "chaossm"
    model = ChaoSSMLM(cfg, phi_kind, VOCAB_SIZE, READOUT_SIZE).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, betas=(0.9, 0.95))

    use_lyap = (arm == "chaossm_corridor")
    lo = hi = None
    if use_lyap:
        lo = cfg.corridor_target - cfg.corridor_half_width
        hi = cfg.corridor_target + cfg.corridor_half_width
    a_hi = a_lo = 0.0
    warmup_steps = int(cfg.warmup_frac * cfg.steps)
    curriculum_steps = cfg.curriculum_frac * cfg.steps
    hist, diverged = [], False

    for step in range(cfg.steps):
        progress = 1.0 if not use_curriculum else min(1.0, step / max(1, curriculum_steps))
        inp, tgt = make_parity_batch(cfg, cfg.batch, DEVICE, gen, progress)
        need_traj = (arm != "linear")
        out = model(inp, return_traj=need_traj)
        logits, traj, xseq = out if need_traj else (out, None, None)
        L_task = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), tgt.reshape(-1),
                                 ignore_index=-100)

        L_pen = torch.zeros((), device=DEVICE); lam_val = float("nan")
        if arm != "linear":
            in_warmup = step < warmup_steps
            lam = lambda_window(model, traj, xseq, cfg.lyap_window, cfg.lyap_subsample)
            lam_val = float(lam.detach())
            if use_lyap and not in_warmup:
                L_pen = a_hi * F.relu(lam - hi) + a_lo * F.relu(lo - lam)

        opt.zero_grad(set_to_none=True); (L_task + L_pen).backward()
        gn = torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip).item()
        opt.step()

        if use_lyap and step >= warmup_steps:
            a_hi = float(np.clip(a_hi + cfg.dual_lr * (lam_val - hi), 0.0, cfg.alpha_max))
            a_lo = float(np.clip(a_lo + cfg.dual_lr * (lo - lam_val), 0.0, cfg.alpha_max))

        if not (math.isfinite(L_task.item()) and math.isfinite(gn)):
            diverged = True; break

        if step % cfg.eval_every == 0:
            with torch.no_grad():
                mask = tgt != -100
                pred = logits.argmax(-1)
                acc = ((pred == tgt) & mask).float().sum().item() / mask.float().sum().clamp(min=1).item()
            hist.append(dict(step=step, arm=arm, curriculum=use_curriculum, seed=seed,
                             loss=L_task.item(), acc=acc, lam=lam_val, gnorm=gn, progress=progress))

    tail = hist[-max(1, len(hist)//5):]
    s = dict(arm=arm, curriculum=use_curriculum, seed=seed, diverged=diverged,
             final_acc=float(np.mean([h["acc"] for h in tail])) if tail else np.nan,
             best_acc=float(np.max([h["acc"] for h in hist])) if hist else np.nan,
             decay_from_peak=float(np.max([h["acc"] for h in hist]) -
                                   np.mean([h["acc"] for h in tail])) if hist else np.nan,
             lam_settled=float(np.nanmean([h["lam"] for h in tail])) if tail else np.nan)
    return s, pd.DataFrame(hist)

## 5. Run — 3 arms × {curriculum, no-curriculum} × 5 seeds

In [7]:
rows, hists = [], []
total = len(CFG.arms) * len(CFG.curricula) * CFG.seeds
done = 0
for use_cur in CFG.curricula:
    for arm in CFG.arms:
        for sd in range(CFG.seeds):
            s, h = train_one(arm, use_cur, sd, CFG)
            rows.append(s); hists.append(h); done += 1
            flag = " DIVERGED" if s["diverged"] else ""
            print(f"[{done}/{total}] curriculum={use_cur!s:5s} {arm:18s} seed={sd}  "
                  f"final={s['final_acc']:.3f} (chance 0.500)  best={s['best_acc']:.3f}  "
                  f"decay={s['decay_from_peak']:.3f}  lam={s['lam_settled']:+.3f}{flag}")

SUM = pd.DataFrame(rows); HIST = pd.concat(hists, ignore_index=True)
savetable(SUM, "phase1b_summary.csv"); savetable(HIST, "phase1b_history.csv")

[1/30] curriculum=True  linear             seed=0  final=0.511 (chance 0.500)  best=0.672  decay=0.161  lam=+nan
[2/30] curriculum=True  linear             seed=1  final=0.482 (chance 0.500)  best=0.672  decay=0.189  lam=+nan
[3/30] curriculum=True  linear             seed=2  final=0.489 (chance 0.500)  best=0.672  decay=0.183  lam=+nan
[4/30] curriculum=True  linear             seed=3  final=0.514 (chance 0.500)  best=0.672  decay=0.157  lam=+nan
[5/30] curriculum=True  linear             seed=4  final=0.506 (chance 0.500)  best=0.641  decay=0.135  lam=+nan
[6/30] curriculum=True  chaossm_free       seed=0  final=0.491 (chance 0.500)  best=0.672  decay=0.181  lam=+0.648
[7/30] curriculum=True  chaossm_free       seed=1  final=0.507 (chance 0.500)  best=0.625  decay=0.118  lam=+0.387
[8/30] curriculum=True  chaossm_free       seed=2  final=0.482 (chance 0.500)  best=0.641  decay=0.158  lam=+0.362
[9/30] curriculum=True  chaossm_free       seed=3  final=0.494 (chance 0.500)  best=0.672 

## 6. Results

Two questions: **did curriculum help** (compare `curriculum=True` vs `False` within each arm),
and, only if it did, **does chaos vs. linear now show a real difference** on a task that
actually learned something.

In [8]:
piv = SUM.pivot_table(index="arm", columns="curriculum", values="final_acc", aggfunc="mean")
piv.columns = [f"curriculum={c}" for c in piv.columns]
print("mean final accuracy (chance = 0.500):")
print(piv.round(4).to_string())
savetable(piv.reset_index(), "phase1b_results_table.csv")

def paired(mask_a, mask_b, col="final_acc"):
    x = SUM[mask_a].sort_values("seed")[col].values
    y = SUM[mask_b].sort_values("seed")[col].values
    m = np.isfinite(x) & np.isfinite(y); x, y = x[m], y[m]
    if len(x) < 3 or np.allclose(x, y):
        return dict(n=len(x), p=np.nan, median_diff=float(np.median(x-y)) if len(x) else np.nan)
    st, p = wilcoxon(x, y)
    return dict(n=len(x), p=float(p), median_diff=float(np.median(x-y)))

print("\ncurriculum vs. no-curriculum, within each arm:")
cur_rows = []
for arm in CFG.arms:
    r = paired(SUM.arm.eq(arm) & SUM.curriculum.eq(True), SUM.arm.eq(arm) & SUM.curriculum.eq(False))
    r["arm"] = arm; cur_rows.append(r)
    print(f"  {arm:18s} n={r['n']} p={r['p']:.4f} median_diff={r['median_diff']:+.4f}")
CUR = pd.DataFrame(cur_rows); savetable(CUR, "phase1b_curriculum_effect.csv")

any_learned = bool(SUM.final_acc.max() > 0.55)
print(f"\nany run clears the 0.55 learnability bar: {any_learned}")

print("\nchaos vs. linear, WITHIN the curriculum setting that worked best:")
best_cur = piv.max(axis=1).idxmax()   # arm with the best mean, informs which curriculum col to trust
best_col = piv.loc[best_cur].idxmax() == piv.columns
use_curriculum_flag = "True" in piv.columns[piv.loc[best_cur].values.argmax()]
chaos_vs_lin = []
for arm in ("chaossm_free", "chaossm_corridor"):
    r = paired(SUM.arm.eq(arm) & SUM.curriculum.eq(use_curriculum_flag),
              SUM.arm.eq("linear") & SUM.curriculum.eq(use_curriculum_flag))
    r["arm"] = arm; r["curriculum"] = use_curriculum_flag; chaos_vs_lin.append(r)
    print(f"  {arm} vs linear (curriculum={use_curriculum_flag}): "
          f"n={r['n']} p={r['p']:.4f} median_diff={r['median_diff']:+.4f}")
CVL = pd.DataFrame(chaos_vs_lin); savetable(CVL, "phase1b_chaos_vs_linear.csv")
print(f"\nmin achievable two-sided p at n={CFG.seeds}: {2/(2**CFG.seeds):.4f}")

mean final accuracy (chance = 0.500):
                  curriculum=False  curriculum=True
arm                                                
chaossm_corridor            0.4950           0.5145
chaossm_free                0.4924           0.4936
linear                      0.5028           0.5004
saved /content/drive/My Drive/ChaoSSM/results/phase1b/phase1b_results_table.csv

curriculum vs. no-curriculum, within each arm:
  linear             n=5 p=0.6250 median_diff=-0.0012
  chaossm_free       n=5 p=1.0000 median_diff=-0.0025
  chaossm_corridor   n=5 p=0.0625 median_diff=+0.0163
saved /content/drive/My Drive/ChaoSSM/results/phase1b/phase1b_curriculum_effect.csv

any run clears the 0.55 learnability bar: False

chaos vs. linear, WITHIN the curriculum setting that worked best:
  chaossm_free vs linear (curriculum=True): n=5 p=0.6250 median_diff=-0.0119
  chaossm_corridor vs linear (curriculum=True): n=5 p=0.3125 median_diff=+0.0044
saved /content/drive/My Drive/ChaoSSM/results/phase1b/

In [9]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for arm in CFG.arms:
    for use_cur, ls in [(True, "-"), (False, "--")]:
        d = HIST[(HIST.arm==arm) & (HIST.curriculum==use_cur)].groupby("step").acc.mean()
        ax[0].plot(d.index, d.values, ls, label=f"{arm} (curr={use_cur})", alpha=.85)
ax[0].axhline(0.5, color="r", ls=":", label="chance")
ax[0].set(xlabel="step", ylabel="parity accuracy", title="Accuracy over training")
ax[0].legend(fontsize=7); ax[0].grid(alpha=.3)

d2 = HIST[HIST.curriculum==True].groupby("step").progress.mean()
ax[1].plot(d2.index, d2.values, color="purple")
ax[1].set(xlabel="step", ylabel="curriculum progress", title="Difficulty schedule (curriculum=True)")
ax[1].grid(alpha=.3)
fig.tight_layout(); savefig(fig, "phase1b_results.png"); plt.close(fig)

saved /content/drive/My Drive/ChaoSSM/results/phase1b/phase1b_results.png


## 7. Verdict

| Observation | Reading |
|---|---|
| `curriculum=True` clears the learnability bar (~0.55+), `False` doesn't | Curriculum fixed the sparse-credit-assignment problem; the fix was necessary, not just more steps. Report parity results only from the curriculum condition. |
| Both conditions clear the bar similarly | Extra steps alone were sufficient; curriculum wasn't the active ingredient — simplifies the story (no need to justify curriculum in the paper). |
| Neither condition clears the bar | Parity remains unlearnable at this scale/budget for this architecture family. Report it honestly as an open limitation rather than pushing further — a Q1 reviewer respects a stated boundary more than a buried failure. |
| Chaos beats linear on parity once it's learnable | This is the result the whole proposal was built to produce — a genuine, mechanistic, statistically supported case that chaos helps specifically on long-range dependency tasks and not on precise-retrieval tasks (assoc_recall). Combined with Phase 1's other three results, this is a complete, nuanced, four-task story ready for the Q1 methods section. |
| Linear ties or beats chaos on parity too, once learnable | The expressivity claim needs real tempering — chaos may not generally help long-range tasks either, only the specific copy-like structure tested so far. Report as a boundary condition on the claim, not a failure of the project. |

In [10]:
verdict = dict(
    smoke_test=SMOKE_TEST,
    results_table=piv.round(4).to_dict("index"),
    any_learned=any_learned,
    curriculum_effect=CUR.to_dict("records"),
    chaos_vs_linear_best_setting=CVL.to_dict("records"),
    config=dict(steps=CFG.steps, seeds=CFG.seeds, curriculum_frac=CFG.curriculum_frac,
                min_gap=CFG.min_gap, parity_len=CFG.parity_len),
)
savejson(verdict, "phase1b_verdict.json")
print(json.dumps(verdict, indent=2, default=str))

saved /content/drive/My Drive/ChaoSSM/results/phase1b/phase1b_verdict.json
{
  "smoke_test": false,
  "results_table": {
    "chaossm_corridor": {
      "curriculum=False": 0.495,
      "curriculum=True": 0.5145
    },
    "chaossm_free": {
      "curriculum=False": 0.4924,
      "curriculum=True": 0.4936
    },
    "linear": {
      "curriculum=False": 0.5028,
      "curriculum=True": 0.5004
    }
  },
  "any_learned": false,
  "curriculum_effect": [
    {
      "n": 5,
      "p": 0.625,
      "median_diff": -0.0012499999999999734,
      "arm": "linear"
    },
    {
      "n": 5,
      "p": 1.0,
      "median_diff": -0.0025000000000000022,
      "arm": "chaossm_free"
    },
    {
      "n": 5,
      "p": 0.0625,
      "median_diff": 0.016250000000000098,
      "arm": "chaossm_corridor"
    }
  ],
  "chaos_vs_linear_best_setting": [
    {
      "n": 5,
      "p": 0.625,
      "median_diff": -0.011874999999999969,
      "arm": "chaossm_free",
      "curriculum": true
    },
    {
      